In [ ]:
#| hide
from programming_first_principles.core import prepare_notebook
from gzip import compress
from io import BytesIO, TextIOWrapper
import sys

In [ ]:
#| hide
prepare_notebook()

# Encoding and Decoding
* **Encoding** is a rule that maps a sequence of abstract Unicode code points (`str`) into a sequence of bytes (`bytes`) for storage at rest or transmission in transit.
* **Decoding** is the inverse rule that reconstructs the sequence of Unicode code points (`str`) from a sequence of bytes (`bytes`).

## Encoding (`str` → `bytes`) and Decoding (`bytes` → `str`)
* Passing a `str` to `bytes(s)` without specifying an encoding fails with `TypeError: string argument without an encoding`.
* `s.encode(encoding)` (or `bytes(s, encoding)`) converts `str` to `bytes`; `b.decode(encoding)` converts `bytes` back to `str`.
* Python's default text encoding (`sys.getdefaultencoding()`) is `'utf-8'`.

#### `bytes(str)` Requires an Encoding

In [ ]:
emoji_string = 'Python 3 - 🐍 🐍 🐍 😊 😠 😆'
try:
    bytes(emoji_string)
except TypeError as e:
    f'{e = }'

"e = TypeError('string argument without an encoding')"

#### `.encode()` and `.decode()` with UTF-8

In [ ]:
f'{sys.getdefaultencoding() = }'

"sys.getdefaultencoding() = 'utf-8'"

In [ ]:
bytes_utf8 = bytes(emoji_string, 'utf-8')
encoded_default = emoji_string.encode()

f'{bytes_utf8 == encoded_default = }'
f'{encoded_default = }'
f'{encoded_default.decode("utf-8") = }'

'bytes_utf8 == encoded_default = True'

"encoded_default = b'Python 3 - \\xf0\\x9f\\x90\\x8d \\xf0\\x9f\\x90\\x8d \\xf0\\x9f\\x90\\x8d \\xf0\\x9f\\x98\\x8a \\xf0\\x9f\\x98\\xa0 \\xf0\\x9f\\x98\\x86'"

'encoded_default.decode("utf-8") = \'Python 3 - 🐍 🐍 🐍 😊 😠 😆\''

#### Variable-Length Encoding in UTF-8
* UTF-8 (designed by Ken Thompson and Rob Pike) is a variable-width encoding: ASCII code points (`U+0000`..`U+007F`) use 1 byte, while supplementary code points like emojis (`U+10000`..`U+10FFFF`) use 4 bytes.

In [ ]:
ascii_char = emoji_string[0]     # 'P'
emoji_char = emoji_string[19]    # '😠'

f'{ascii_char = }, {ord(ascii_char) = }, {ascii_char.encode("utf-8") = }, {len(ascii_char.encode("utf-8")) = }'
f'{emoji_char = }, {ord(emoji_char) = }, {emoji_char.encode("utf-8") = }, {len(emoji_char.encode("utf-8")) = }'
f'{len(emoji_string) = } code points vs {len(emoji_string.encode("utf-8")) = } UTF-8 bytes'

'ascii_char = \'P\', ord(ascii_char) = 80, ascii_char.encode("utf-8") = b\'P\', len(ascii_char.encode("utf-8")) = 1'

'emoji_char = \'😠\', ord(emoji_char) = 128544, emoji_char.encode("utf-8") = b\'\\xf0\\x9f\\x98\\xa0\', len(emoji_char.encode("utf-8")) = 4'

'len(emoji_string) = 22 code points vs len(emoji_string.encode("utf-8")) = 40 UTF-8 bytes'

#### Decoding a Sequence of Integer Byte Values

In [ ]:
bytes_sequence = bytes([
    80, 121, 116, 104, 111, 110, 32, 51, 32, 45, 32,
    240, 159, 144, 141, 32, 240, 159, 144, 141, 32, 240, 159, 144, 141, 32,
    240, 159, 152, 138, 32, 240, 159, 152, 160, 32, 240, 159, 152, 134
])
decoded_str = bytes_sequence.decode('utf-8')
f'{decoded_str = }'

"decoded_str = 'Python 3 - 🐍 🐍 🐍 😊 😠 😆'"

In [ ]:
f'{sys.getsizeof(bytes_sequence) = }, {sys.getsizeof(decoded_str) = }'

'sys.getsizeof(bytes_sequence) = 73, sys.getsizeof(decoded_str) = 148'

## Encoding is Out-of-Band
* Raw bytes do not store which encoding produced them; the encoding must be known out-of-band.
* Encoding a character that does not exist in the target character set (such as emojis in `ascii` or `iso-8859-1`) raises `UnicodeEncodeError` unless an `errors` policy (`'ignore'`, `'replace'`) is specified.
* Decoding bytes with the wrong encoding either raises `UnicodeDecodeError` or silently produces garbled text (*mojibake*).

#### `UnicodeEncodeError` and Error Handlers (`ignore`, `replace`)

In [ ]:
my_string = 'Python 3 - 🙂 😠 😀'

try:
    my_string.encode('ascii')
except UnicodeEncodeError as e:
    f'{e = }'

"e = UnicodeEncodeError('ascii', 'Python 3 - 🙂 😠 😀', 11, 12, 'ordinal not in range(128)')"

In [ ]:
try:
    my_string.encode('iso-8859-1')
except UnicodeEncodeError as e:
    f'{e = }'

"e = UnicodeEncodeError('latin-1', 'Python 3 - 🙂 😠 😀', 11, 12, 'ordinal not in range(256)')"

In [ ]:
f'{my_string.encode(encoding="iso-8859-1", errors="ignore") = }'

'my_string.encode(encoding="iso-8859-1", errors="ignore") = b\'Python 3 -   \''

In [ ]:
f'{my_string.encode(encoding="ascii", errors="replace") = }'

'my_string.encode(encoding="ascii", errors="replace") = b\'Python 3 - ? ? ?\''

#### Decoding with the Wrong Encoding (Mojibake vs `UnicodeDecodeError`)

In [ ]:
try:
    b'\xed'.decode('utf-8')
except UnicodeDecodeError as e:
    f'{e = }'

raw_stream = my_string.encode('utf-8')
read_as_iso = TextIOWrapper(BytesIO(raw_stream), encoding='iso-8859-1').read()
read_as_utf8 = TextIOWrapper(BytesIO(raw_stream), encoding='utf-8').read()

f'{read_as_iso = }'
f'{read_as_utf8 = }'

"e = UnicodeDecodeError('utf-8', b'\\xed', 0, 1, 'unexpected end of data')"

"read_as_iso = 'Python 3 - ð\\x9f\\x99\\x82 ð\\x9f\\x98\\xa0 ð\\x9f\\x98\\x80'"

"read_as_utf8 = 'Python 3 - 🙂 😠 😀'"

## Bytes at the I/O and Compression Boundary
* Operations on binary data (network sockets, file hashes, compression) operate on `bytes` (or buffer-protocol objects like `bytearray` and `memoryview`), not `str`.
* Gzip compress 

In [ ]:
try:
    compress('python3')
except TypeError as e:
    f'{e = }'

compressed = compress('python3'.encode('utf-8'))
f'{compressed = }'
f'{compress(b"python3") == compressed = }'

'e = TypeError("a bytes-like object is required, not \'str\'")'

"compressed = b'\\x1f\\x8b\\x08\\x00\\x14\\xf6\\xdaj\\x02\\xff+\\xa8,\\xc9\\xc8\\xcf3\\x06\\x00\\x9da\\xcc\\xa8\\x07\\x00\\x00\\x00'"

'compress(b"python3") == compressed = True'

## References
* [Ken Thompson & Rob Pike – UTF-8 (1993 USENIX Paper)](https://doc.cat-v.org/plan_9/4th_edition/papers/utf)
* [Python Unicode HOWTO (A. M. Kuchling)](https://docs.python.org/3/howto/unicode.html)
* [Python Library Reference – `codecs` and Standard Encodings](https://docs.python.org/3/library/codecs.html)